In [1]:
import time
import os
import joblib
import pandas as pd
import numpy as np
from keras.models import load_model
from bayes_opt import BayesianOptimization

In [4]:
# ----------------------------------------------------
# 1. 앙상블 모델 및 스케일러 로딩 (PV 모듈 맞춤형)
# ----------------------------------------------------
N_ENSEMBLE = 10
TARGET_NAMES = ["Deflection(mm)", "Weight(kg)"]
MODEL_PATHS = [f'saved_models/pv_ensemble_model_{i+1}.keras' for i in range(N_ENSEMBLE)]
SCALER_DIR = '.' # 스케일러가 저장된 디렉토리 (필요시 경로 수정)

# 앙상블 모델 로딩 
try:
    ENSEMBLE_MODELS = [load_model(path) for path in MODEL_PATHS]
except Exception as e:
    print(f"오류: 앙상블 모델 로드 실패. 저장 경로 확인 필요. ({e})")
    ENSEMBLE_MODELS = [] 

# 저장된 스케일러 로드 (이전 단계에서 X, y 분리 스케일링 적용 가정)
try:
    X_SCALER_LOADED = joblib.load(os.path.join(SCALER_DIR, 'scaler_X_pvmodule.joblib')) 
    Y_SCALER_LOADED = joblib.load(os.path.join(SCALER_DIR, 'scaler_y_pvmodule.joblib')) 
    
    # RobustScaler의 scale 인자 추출 (표준편차 복원용)
    Y_STDS = Y_SCALER_LOADED.scale_
except Exception as e:
    print(f"오류: 스케일러 로드 실패. 파일 경로 및 joblib 저장 확인 필요. ({e})")
    raise

print("앙상블 모델 및 스케일러 로딩 완료.")

앙상블 모델 및 스케일러 로딩 완료.


C:\Users\admin\anaconda3\envs\py31010_auto\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.5.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [27]:
# ----------------------------------------------------
# 2. 최적 인자 탐색을 위한 Objective 함수
# ----------------------------------------------------
CONSTRAINTS = {
    "Deflection(mm)": {"min": None, "max": 12.3}, 
    "Weight(kg)": {"min": None, "max": 3.6}       
}

PENALTY_CONST = 100.0

In [28]:
def objective_for_optimization_constrained(a, b, c, d, e):
    # 1. 알파벳 순서대로 정상 입력 (a, b, c, d, e)
    X_input = np.array([[a, b, c, d, e]])
    X_scaled = X_SCALER_LOADED.transform(X_input)
    
    # 2. 앙상블 평균 예측 (현재 모델은 **원본(Raw) 물리적 값**을 뱉어냄)
    preds_raw = []
    if not ENSEMBLE_MODELS: return -100
    for model in ENSEMBLE_MODELS:
        preds_raw.append(model.predict(X_scaled, verbose=0)[0])
    
    y_pred_mean_raw = np.mean(np.array(preds_raw), axis=0)

    # 3. F_obj를 공평하게 계산하기 위해 예측값을 스케일링
    # (처짐량과 무게의 단위 크기가 다르므로 스케일 변환 후 5:5 비율 적용)
    y_pred_mean_scaled = Y_SCALER_LOADED.transform(y_pred_mean_raw.reshape(1, -1))[0]
    
    # 4. 목적 함수 (F_obj) 계산 
    # 변위(음수)는 0에 가깝도록 "최대화(+1)", 무게(양수)는 "최소화(-1)"
    GOAL_SIGNS = np.array([1, -1]) 
    WEIGHTS = np.array([1.0, 1.0]) 
    
    F_obj_terms = GOAL_SIGNS * WEIGHTS * y_pred_mean_scaled
    F_obj = np.sum(F_obj_terms)
    
    # 5. 제약 조건 검사 및 벌칙 적용 (벌칙은 실제 물리적 값으로 철저히 검사)
    penalty = 0.0
    for i, target_name in enumerate(TARGET_NAMES):
        value = y_pred_mean_raw[i] 
        
        # 처짐량은 절댓값으로 변환하여 검사
        if target_name == "Deflection(mm)":
            check_value = abs(value)
        else:
            check_value = value
                
        if target_name in CONSTRAINTS:
            max_req = CONSTRAINTS[target_name]["max"]
            if max_req is not None and check_value > max_req:
                penalty += PENALTY_CONST * (check_value - max_req)
    
    F_obj_constrained = F_obj - penalty
    return F_obj_constrained

# ----------------------------------------------------
# 3. Bayesian Optimization 실행
# ----------------------------------------------------
start = time.time()

# 물리적 탐색 공간 정의
INPUT_PBOX = {
    'a': (1.5, 2.5), 
    'b': (8.0, 14.0), 
    'c': (1.5, 2.5), 
    'd': (1.7, 3.0),
    'e': (2.5, 5.0)
}

optimizer_constrained = BayesianOptimization(
    f=objective_for_optimization_constrained,
    pbounds=INPUT_PBOX,
    random_state=42 
)

print(f"\n--- 제약 조건 반영 최적 입력 인자 탐색 시작 (Total 450회) ---")
optimizer_constrained.maximize(init_points=50, n_iter=400)


--- 제약 조건 반영 최적 입력 인자 탐색 시작 (Total 450회) ---
|   iter    |  target   |     a     |     b     |     c     |     d     |     e     |
-------------------------------------------------------------------------------------
| 1         | -20.12025 | 1.8745401 | 13.704285 | 2.2319939 | 2.4782560 | 2.8900466 |
| 2         | -25.59292 | 1.6559945 | 8.3485016 | 2.3661761 | 2.4814495 | 4.2701814 |
| 3         | -0.702229 | 1.5205844 | 13.819459 | 2.3324426 | 1.9760408 | 2.9545624 |
| 4         | -3.839992 | 1.6834045 | 9.8254534 | 2.0247564 | 2.2615285 | 3.2280728 |
| 5         | -12.07207 | 2.1118528 | 8.8369631 | 1.7921446 | 2.1762703 | 3.6401749 |
| 6         | -24.26763 | 2.2851759 | 9.1980426 | 2.0142344 | 2.4701389 | 2.6161260 |
| 7         | -30.09517 | 2.1075448 | 9.0231447 | 1.5650515 | 2.9335511 | 4.9140800 |
| 8         | -21.87106 | 2.3083973 | 9.8276826 | 1.5976721 | 2.5895029 | 3.6003812 |
| 9         | 0.2637338 | 1.6220382 | 10.971061 | 1.5343885 | 2.8821165 | 3.1469499 |
| 10    

In [29]:
# ----------------------------------------------------
# 4. 최적 입력 인자 출력 및 결과 검증
# ----------------------------------------------------
best_X_params = optimizer_constrained.max['params']
max_F_obj = optimizer_constrained.max['target']

print("\n=== 제약 조건 만족 복합 목표 최대화 최적 입력 인자 조합 ===")
for k, v in best_X_params.items():
    print(f"    {k}: {v:.4f}")

print(f"최대 목적 함수 값 (F_obj): {max_F_obj:.4f}")

# 입력 순서 원상 복구 (a, b, c, d, e) 및 역변환 제거 반영
X_best_input = np.array([[best_X_params['a'], best_X_params['b'], 
                          best_X_params['c'], best_X_params['d'], best_X_params['e']]])
X_best_scaled = X_SCALER_LOADED.transform(X_best_input)
preds_raw = [model.predict(X_best_scaled, verbose=0)[0] for model in ENSEMBLE_MODELS]

# 모델이 원본 값을 주므로 inverse_transform 삭제
y_best_original = np.mean(np.array(preds_raw), axis=0) 

print("\n--- 최적 해의 예측된 출력값 (Y) ---")
for i, name in enumerate(TARGET_NAMES):
    print(f"    {name}: {y_best_original[i]:.4f}")

print("Elapsed time :", time.time() - start)


=== 제약 조건 만족 복합 목표 최대화 최적 입력 인자 조합 ===
    a: 1.5000
    b: 14.0000
    c: 1.5000
    d: 3.0000
    e: 3.0889
최대 목적 함수 값 (F_obj): 0.6414

--- 최적 해의 예측된 출력값 (Y) ---
    Deflection(mm): -11.3302
    Weight(kg): 3.5496
Elapsed time : 471.8559830188751


In [34]:
# ----------------------------------------------------
# 5. 딥 앙상블의 불확실성 분석 (X_best 지점)
# ----------------------------------------------------
# 변수명이 preds_scaled로 넘어왔더라도, 실제로는 모델이 원본(Raw) 값을 뱉어냅니다.
preds_raw_arr = np.array(preds_scaled) # Shape: (N_ENSEMBLE, 2)

# ★ 수정 핵심: 역변환(inverse_transform)과 scale_ 곱하기를 모두 삭제!
mean_original = np.mean(preds_raw_arr, axis=0)
std_original = np.std(preds_raw_arr, axis=0)

print("\n=== 앙상블 예측 신뢰성 분석 (X_best 지점) ===")
print("  (95% 예측 구간: Mean ± 1.96 * Std)")
results_df = pd.DataFrame({
    'Output': TARGET_NAMES,
    'Mean (Pred)': mean_original,
    'Std (Uncertainty)': std_original,
    'Lower 95% PI': mean_original - 1.96 * std_original,
    'Upper 95% PI': mean_original + 1.96 * std_original
})
print(results_df.to_markdown(floatfmt=".4f"))

# ----------------------------------------------------
# 6. 국소적 검증 (Local Verification: ±1% Perturbation)
# ----------------------------------------------------
N_LOCAL_SAMPLES = 10 
TOLERANCE_FACTOR = 0.01 
local_f_obj_values = []

for i in range(N_LOCAL_SAMPLES):
    X_local_sample = {}
    for key in ['a', 'b', 'c', 'd', 'e']:
        mean_val = best_X_params[key]
        low = mean_val * (1 - TOLERANCE_FACTOR)
        high = mean_val * (1 + TOLERANCE_FACTOR)
        X_local_sample[key] = np.random.uniform(low, high)
        
    f_obj_val = objective_for_optimization_constrained(
        X_local_sample['a'], X_local_sample['b'], X_local_sample['c'], 
        X_local_sample['d'], X_local_sample['e']
    )
    local_f_obj_values.append(f_obj_val)

print("\n=== 국소적 강건성 검증 (Local Verification ±1%) ===")
print(f"기준점 (X_best) F_obj: {max_F_obj:.4f}")
print(f"주변 샘플 ({N_LOCAL_SAMPLES}개) F_obj 범위: {np.min(local_f_obj_values):.4f} ~ {np.max(local_f_obj_values):.4f}")
print(f"주변 샘플 F_obj 평균: {np.mean(local_f_obj_values):.4f}")


=== 앙상블 예측 신뢰성 분석 (X_best 지점) ===
  (95% 예측 구간: Mean ± 1.96 * Std)
|    | Output         |   Mean (Pred) |   Std (Uncertainty) |   Lower 95% PI |   Upper 95% PI |
|---:|:---------------|--------------:|--------------------:|---------------:|---------------:|
|  0 | Deflection(mm) |      -10.6214 |              0.0936 |       -10.8048 |       -10.4380 |
|  1 | Weight(kg)     |        4.0122 |              0.0411 |         3.9315 |         4.0928 |

=== 국소적 강건성 검증 (Local Verification ±1%) ===
기준점 (X_best) F_obj: 0.6414
주변 샘플 (10개) F_obj 범위: 0.6186 ~ 0.6456
주변 샘플 F_obj 평균: 0.6337
